In [1]:
import os
from dotenv import load_dotenv

load_dotenv()  # this is used to load all the packages installed in env

True

In [2]:
# all these below import packages are been installed in pyproject.toml file when we ran the uv add ...
from langchain_groq import ChatGroq



In [3]:
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

In [4]:
response =llm.invoke("what is famous places to visit in vishapatnam for 2 day trip ?")
response.content

"Visakhapatnam, also known as Vizag, is a beautiful coastal city in the state of Andhra Pradesh, India. Here are some famous places to visit in Visakhapatnam for a 2-day trip:\n\n**Day 1:**\n\n1. **Ramakrishna Beach**: A popular beach in Visakhapatnam, known for its scenic views and vibrant atmosphere.\n2. **Kailasagiri Hill**: A hilltop park with stunning views of the city and the Bay of Bengal.\n3. **Submarine Museum**: A unique museum showcasing a Russian submarine, the INS Kursura, which was decommissioned in 2001.\n4. **Visakha Museum**: A museum showcasing the history and culture of Visakhapatnam, with exhibits on the city's ancient past and its role in the Indian independence movement.\n5. **Dolphin's Nose**: A scenic viewpoint with stunning views of the Bay of Bengal and the surrounding coastline.\n\n**Day 2:**\n\n1. **Simhachalam Temple**: A beautiful temple dedicated to Lord Vishnu, known for its stunning architecture and vibrant festivals.\n2. **Borra Caves**: A network of c

## RAG IMPLEMENTATION WITH YOUR OWN TEXT DATA

# Step 1: Preparing Document for your own text

In [5]:
from langchain_core.documents import Document

# my text here
my_text = """
Visakhapatnam, also known as Vizag, is a beautiful coastal city in the state of Andhra Pradesh, India. Here are some famous places to visit 
in Visakhapatnam for a 2-day trip:\n\n**Day 1:**\n\n1. **Ramakrishna Beach**: A popular beach in Visakhapatnam, known for its scenic views and vibrant 
atmosphere.\n2. **Kailasagiri Hill**: A hilltop park with stunning views of the city and the Bay of Bengal.\n3. **Submarine Museum**: A unique museum 
showcasing a Russian submarine, the INS Kursura, which was decommissioned in 2001.\n4. **Visakha Museum**: A museum showcasing the history and culture of 
Visakhapatnam, with exhibits on the city's ancient past and its role in the Indian independence movement.\n5. **Dolphin's Nose**: A scenic viewpoint with stunning 
views of the Bay of Bengal and the surrounding coastline.\n\n**Day 2:**\n\n1. **Borra Caves**: A network of limestone caves located about 90 km from Visakhapatnam, 
known for their stunning rock formations and ancient fossils.\n2. **Araku Valley**: A scenic valley located about 115 km from Visakhapatnam, known for its stunning
 natural beauty, waterfalls, and tribal villages.\n3. **Thotlakonda Buddhist Stupa**: A ancient Buddhist stupa located on a hilltop, with stunning views of the 
 surrounding countryside.\n4. **Simhachalam Temple**: A beautiful temple dedicated to Lord Vishnu, known for its stunning architecture and vibrant festivals.
 \n5. **Rushikonda Beach**: A scenic beach located about 8 km from Visakhapatnam, known for its stunning views and water sports.\n\n**Other attractions:**\n\n* 
 **Visakhapatnam Steel Plant**: A major steel plant in India, known for its massive production capacity and innovative technology.\n* **Indira Gandhi Zoological 
 Park**: A zoo located in the heart of the city, with a wide range of animals and birds.\n* **Kambalakonda Wildlife Sanctuary**: A wildlife sanctuary located about 
 20 km from Visakhapatnam, known for its stunning natural beauty and diverse wildlife.\n\n**Tips:**\n\n* Visakhapatnam has a tropical climate, with hot summers and 
 mild winters. The best time to visit is from October to February.\n* The city has a well-developed transportation system, with buses, taxis, and auto-rickshaws 
 available.\n* Be sure to try the local cuisine, which includes popular dishes like fish fry, chicken tikka, and idlis.\n* Visakhapatnam is a safe city, but be 
 sure to take necessary precautions to avoid petty theft and scams
"""

# we will create a doc your your text
doc = [Document(page_content=my_text, metadata={"source": "ABC", "doc_id": "123"})]  # passing doc + metadata

## Step 2: Splitting the documents into chunks

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, # this is the size of each chunk
                                        chunk_overlap=50 # this we need suppose the 1st chunk took half of the text and 2nd half is in another chunk , so it doesnot make sense ,so overlapping will overlap the first chunk data of 50 char into 2nd chunk.
                                        )

chunks = splitter.split_documents(doc) # this will split the doc into chunks 

## Step 3: Create Embeddings for these chunks

In [ ]:
from google import genai
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
for model in client.models.list():
    if "embed" in model.name.lower():
        print(model.name)

models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2


In [14]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embedding_model = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

### Step 4: Create and store Embeddings in Vector Store

In [15]:
from langchain_community.vectorstores import FAISS # FAISS DB is a free vector store

vectorstore = FAISS.from_documents(documents=chunks, embedding=embedding_model)

### Step 5: Semantic Search

In [17]:
vectorstore.similarity_search("which is the famous beach in visakhapatnam ?",k=2)

[Document(id='ef3b2328-06e0-4149-868c-9d3d8c8421a5', metadata={'source': 'ABC', 'doc_id': '123'}, page_content='1. **Ramakrishna Beach**: A popular beach in Visakhapatnam, known for its scenic views and vibrant \natmosphere.\n2. **Kailasagiri Hill**: A hilltop park with stunning views of the city and the Bay of Bengal.\n3. **Submarine Museum**: A unique museum \nshowcasing a Russian submarine, the INS Kursura, which was decommissioned in 2001.\n4. **Visakha Museum**: A museum showcasing the history and culture of'),
 Document(id='1f4fa671-43f3-4c3b-a11d-da5ec587cbbb', metadata={'source': 'ABC', 'doc_id': '123'}, page_content='Visakhapatnam, also known as Vizag, is a beautiful coastal city in the state of Andhra Pradesh, India. Here are some famous places to visit \nin Visakhapatnam for a 2-day trip:\n\n**Day 1:**')]

## Talk to LLM

In [19]:
context = vectorstore.similarity_search("which is the famous beach in visakhapatnam ?",k=2)
response = llm.invoke(f"Answer the question based on the following context: {context}")
print(response.content)

What are some famous places to visit in Visakhapatnam for a 2-day trip?

Based on the provided context, here are some famous places to visit in Visakhapatnam for a 2-day trip:

1. **Ramakrishna Beach**: A popular beach in Visakhapatnam, known for its scenic views and vibrant atmosphere.
2. **Kailasagiri Hill**: A hilltop park with stunning views of the city and the Bay of Bengal.
3. **Submarine Museum**: A unique museum showcasing a Russian submarine, the INS Kursura, which was decommissioned in 2001.
4. **Visakha Museum**: A museum showcasing the history and culture of Visakhapatnam.

These places are mentioned in the provided context as popular tourist attractions in Visakhapatnam.
